# Step 1 — Download Flint GGUF and publish it as a Kaggle Dataset

This notebook downloads the official `ggml-org/Flint-GGUF` **Q4_K_M** file and uploads it as a private Kaggle Dataset. Once uploaded, the model persists in Kaggle and can be attached to any later inference notebook; you do **not** need to download it again for each server session.

**Why `/tmp`?** The model file is about 23.4 GB, larger than the usual 20 GB `/kaggle/working` output allowance. `/tmp` is only temporary staging during this upload. The reusable copy is the Kaggle Dataset created at the end.

### Before running
- Enable **Internet** in the notebook settings.
- Set `KAGGLE_USERNAME` in Cell 1.
- Configure Kaggle API credentials. The notebook can read a Kaggle Secret named `KAGGLE_KEY`; if available, it also reads `KAGGLE_USERNAME`. Alternatively, make `~/.kaggle/kaggle.json` available.
- This notebook creates a **private** dataset by default.
- Source model card: https://huggingface.co/ggml-org/Flint-GGUF


# Step 1 — Download Flint GGUF and publish it as a Kaggle Dataset

This notebook downloads the official `ggml-org/Flint-GGUF` **Q4_K_M** file and uploads it as a private Kaggle Dataset. Once uploaded, the model persists in Kaggle and can be attached to any later inference notebook; you do **not** need to download it again for each server session.

**Why `/tmp`?** The model file is about 23.4 GB, larger than the usual 20 GB `/kaggle/working` output allowance. `/tmp` is only temporary staging during this upload. The reusable copy is the Kaggle Dataset created at the end.

### Before running
- Enable **Internet** in the notebook settings.
- Set `KAGGLE_USERNAME` in Cell 1.
- Configure Kaggle API credentials. The notebook can read a Kaggle Secret named `KAGGLE_KEY`; if available, it also reads `KAGGLE_USERNAME`. Alternatively, make `~/.kaggle/kaggle.json` available.
- This notebook creates a **private** dataset by default.
- Source model card: https://huggingface.co/ggml-org/Flint-GGUF


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

KAGGLE_USERNAME = "ender6831"   # e.g. "johndoe"
DATASET_SLUG = "flint-gguf"

HF_REPO = "ggml-org/Flint-GGUF"
MODEL_FILE = "Flint-Q4_K_M.gguf"

# Upload an update if the dataset already exists; otherwise create it.
# Set True only after you've created this dataset once.
UPDATE_EXISTING_DATASET = False

# Temporary staging path: the final reusable copy lives in the Kaggle Dataset.
STAGING_DIR = "/tmp/flint-gguf-dataset"

# ══════════════════════════════════════════════════════════════════════════════
import os

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Set KAGGLE_USERNAME in this cell or as an environment variable.")

DATASET_ID = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
print("Hugging Face repository:", HF_REPO)
print("Model file:", MODEL_FILE)
print("Kaggle Dataset:", DATASET_ID)
print("Update existing dataset:", UPDATE_EXISTING_DATASET)


In [ ]:
# Install required tools
%pip -q install -U huggingface_hub kaggle


In [ ]:
# Download the exact model file into temporary staging
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, os

staging = Path(STAGING_DIR)
staging.mkdir(parents=True, exist_ok=True)
model_path = staging / MODEL_FILE

if not model_path.exists() or model_path.stat().st_size < 1_000_000_000:
    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=MODEL_FILE,
        local_dir=STAGING_DIR,
    )
    downloaded_path = Path(downloaded_path)
    if downloaded_path.resolve() != model_path.resolve():
        shutil.copy2(downloaded_path, model_path)

size_gb = model_path.stat().st_size / 1e9
print(f"Downloaded: {model_path}")
print(f"Size: {size_gb:.2f} GB")
assert size_gb > 10, "Downloaded file is unexpectedly small; stop and check the download."


In [ ]:
# Write Kaggle Dataset metadata
import json
from pathlib import Path

metadata = {
    "title": "Flint GGUF Q4_K_M",
    "id": DATASET_ID,
    "licenses": [{"name": "apache-2.0"}],
    "keywords": ["flint", "gguf", "llm", "llama-cpp"],
}

(staging / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Dataset folder is ready:")
for f in sorted(staging.iterdir()):
    if f.is_file():
        print(f" - {f.name}: {f.stat().st_size / 1e9:.2f} GB")


In [ ]:
# Configure Kaggle credentials
# Preferred: add a Kaggle Notebook Secret named KAGGLE_KEY.
# The Kaggle username can be configured above or stored as KAGGLE_USERNAME.
import os, json
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    key = secrets.get_secret("KAGGLE_KEY")
    if key:
        os.environ["KAGGLE_KEY"] = key
    try:
        secret_username = secrets.get_secret("KAGGLE_USERNAME")
        if secret_username:
            os.environ["KAGGLE_USERNAME"] = secret_username
    except Exception:
        pass
except Exception as e:
    print("Kaggle Secrets unavailable in this environment:", type(e).__name__)

api_key = os.environ.get("KAGGLE_KEY")
api_username = os.environ.get("KAGGLE_USERNAME") or KAGGLE_USERNAME

if api_key:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    credentials = kaggle_dir / "kaggle.json"
    credentials.write_text(json.dumps({"username": api_username, "key": api_key}))
    credentials.chmod(0o600)
    print("Kaggle API credentials configured.")
elif (Path.home() / ".kaggle" / "kaggle.json").exists():
    print("Using existing ~/.kaggle/kaggle.json.")
else:
    raise RuntimeError(
        "Kaggle API credentials not found. Add a Kaggle Secret named KAGGLE_KEY "
        "or place your kaggle.json at ~/.kaggle/kaggle.json."
    )


In [ ]:
# Upload the model as a reusable Kaggle Dataset
# --dir-mode skip avoids making a second giant ZIP archive of the 23+ GB model.
import subprocess

if UPDATE_EXISTING_DATASET:
    cmd = [
        "kaggle", "datasets", "version",
        "-p", STAGING_DIR,
        "-m", "Update Flint GGUF Q4_K_M",
        "-r", "skip",
    ]
else:
    cmd = [
        "kaggle", "datasets", "create",
        "-p", STAGING_DIR,
        "-r", "skip",
    ]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Kaggle Dataset upload failed. Check the error above, the available disk space, "
        "your API credentials, and whether the dataset already exists. If it already "
        "exists, set UPDATE_EXISTING_DATASET=True in Cell 1 and rerun from this cell."
    )

print("\nUpload command completed.")
print("Reusable dataset ID:", DATASET_ID)


## After upload

Attach `YOUR_KAGGLE_USERNAME/flint-gguf` to the inference notebook as an Input Dataset. The model file will be available from Kaggle's mounted input directory and will survive the end of the current notebook session.

Flint is **text-only**: there is no `mmproj` file to download or pass to `llama-server`. The Q4_K_M file is listed as about 23.4 GB by the model repository. See the model card for details: https://huggingface.co/ggml-org/Flint-GGUF


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — Edit this cell only
# ══════════════════════════════════════════════════════════════════════════════

KAGGLE_USERNAME = "ender6831"   # e.g. "johndoe"
DATASET_SLUG = "flint-gguf"

HF_REPO = "ggml-org/Flint-GGUF"
MODEL_FILE = "Flint-Q4_K_M.gguf"

# Upload an update if the dataset already exists; otherwise create it.
# Set True only after you've created this dataset once.
UPDATE_EXISTING_DATASET = False

# Temporary staging path: the final reusable copy lives in the Kaggle Dataset.
STAGING_DIR = "/tmp/flint-gguf-dataset"

# ══════════════════════════════════════════════════════════════════════════════
import os

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = os.environ.get("KAGGLE_USERNAME", "")
if not KAGGLE_USERNAME:
    raise ValueError("Set KAGGLE_USERNAME in this cell or as an environment variable.")

DATASET_ID = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"
print("Hugging Face repository:", HF_REPO)
print("Model file:", MODEL_FILE)
print("Kaggle Dataset:", DATASET_ID)
print("Update existing dataset:", UPDATE_EXISTING_DATASET)


In [ ]:
# Install required tools
%pip -q install -U huggingface_hub kaggle


In [ ]:
# Download the exact model file into temporary staging
from huggingface_hub import hf_hub_download
from pathlib import Path
import shutil, os

staging = Path(STAGING_DIR)
staging.mkdir(parents=True, exist_ok=True)
model_path = staging / MODEL_FILE

if not model_path.exists() or model_path.stat().st_size < 1_000_000_000:
    downloaded_path = hf_hub_download(
        repo_id=HF_REPO,
        filename=MODEL_FILE,
        local_dir=STAGING_DIR,
    )
    downloaded_path = Path(downloaded_path)
    if downloaded_path.resolve() != model_path.resolve():
        shutil.copy2(downloaded_path, model_path)

size_gb = model_path.stat().st_size / 1e9
print(f"Downloaded: {model_path}")
print(f"Size: {size_gb:.2f} GB")
assert size_gb > 10, "Downloaded file is unexpectedly small; stop and check the download."


In [ ]:
# Write Kaggle Dataset metadata
import json
from pathlib import Path

metadata = {
    "title": "Flint GGUF Q4_K_M",
    "id": DATASET_ID,
    "licenses": [{"name": "apache-2.0"}],
    "keywords": ["flint", "gguf", "llm", "llama-cpp"],
}

(staging / "dataset-metadata.json").write_text(
    json.dumps(metadata, indent=2),
    encoding="utf-8",
)

print("Dataset folder is ready:")
for f in sorted(staging.iterdir()):
    if f.is_file():
        print(f" - {f.name}: {f.stat().st_size / 1e9:.2f} GB")


In [ ]:
# Configure Kaggle credentials
# Preferred: add a Kaggle Notebook Secret named KAGGLE_KEY.
# The Kaggle username can be configured above or stored as KAGGLE_USERNAME.
import os, json
from pathlib import Path

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    key = secrets.get_secret("KAGGLE_KEY")
    if key:
        os.environ["KAGGLE_KEY"] = key
    try:
        secret_username = secrets.get_secret("KAGGLE_USERNAME")
        if secret_username:
            os.environ["KAGGLE_USERNAME"] = secret_username
    except Exception:
        pass
except Exception as e:
    print("Kaggle Secrets unavailable in this environment:", type(e).__name__)

api_key = os.environ.get("KAGGLE_KEY")
api_username = os.environ.get("KAGGLE_USERNAME") or KAGGLE_USERNAME

if api_key:
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    credentials = kaggle_dir / "kaggle.json"
    credentials.write_text(json.dumps({"username": api_username, "key": api_key}))
    credentials.chmod(0o600)
    print("Kaggle API credentials configured.")
elif (Path.home() / ".kaggle" / "kaggle.json").exists():
    print("Using existing ~/.kaggle/kaggle.json.")
else:
    raise RuntimeError(
        "Kaggle API credentials not found. Add a Kaggle Secret named KAGGLE_KEY "
        "or place your kaggle.json at ~/.kaggle/kaggle.json."
    )


In [ ]:
# Upload the model as a reusable Kaggle Dataset
# --dir-mode skip avoids making a second giant ZIP archive of the 23+ GB model.
import subprocess

if UPDATE_EXISTING_DATASET:
    cmd = [
        "kaggle", "datasets", "version",
        "-p", STAGING_DIR,
        "-m", "Update Flint GGUF Q4_K_M",
        "-r", "skip",
    ]
else:
    cmd = [
        "kaggle", "datasets", "create",
        "-p", STAGING_DIR,
        "-r", "skip",
    ]

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout)
if result.stderr:
    print(result.stderr)

if result.returncode != 0:
    raise RuntimeError(
        "Kaggle Dataset upload failed. Check the error above, the available disk space, "
        "your API credentials, and whether the dataset already exists. If it already "
        "exists, set UPDATE_EXISTING_DATASET=True in Cell 1 and rerun from this cell."
    )

print("\nUpload command completed.")
print("Reusable dataset ID:", DATASET_ID)


## After upload

Attach `YOUR_KAGGLE_USERNAME/flint-gguf` to the inference notebook as an Input Dataset. The model file will be available from Kaggle's mounted input directory and will survive the end of the current notebook session.

Flint is **text-only**: there is no `mmproj` file to download or pass to `llama-server`. The Q4_K_M file is listed as about 23.4 GB by the model repository. See the model card for details: https://huggingface.co/ggml-org/Flint-GGUF
